# Dự báo nhu cầu cấp SKU cho xưởng — Pipeline PySpark
**Môn:** Nghiên cứu Dữ liệu lớn & Ứng dụng trong Kinh doanh — TS. Nguyễn Thôn Dã
**Case:** Grace's Sweet (đồ thể thao, sản xuất theo đơn)

Notebook chạy trên **Google Colab**. Luồng: đọc đơn hàng → tách mã hàng (style–màu–size) →
tổng hợp tuần → tạo đặc trưng → GBTRegressor (Spark MLlib) so với 2 baseline →
chọn mô hình tốt nhất → xuất **kế hoạch cắt tuần tới** theo màu/size.

## Bước 0 — Cài đặt môi trường Spark trên Colab
Chạy ô dưới (khoảng 1–2 phút). Dùng `pyspark==3.5.3` để tương thích Java sẵn có của Colab.

In [ ]:
!pip install -q pyspark==3.5.3 openpyxl

## Bước 1 — Tải file đơn hàng lên
Chạy ô dưới, bấm **Choose Files** và chọn file Excel đơn hàng của bạn (ví dụ `orders.xlsx`).

In [ ]:
from google.colab import files
up = files.upload()
XLSX = list(up.keys())[0]     # tên file vừa upload
OUT  = "."                    # lưu kết quả tại thư mục hiện tại
TOP_N, TEST_WEEKS = 30, 12    # dự báo 30 style bán chạy nhất; 12 tuần cuối để kiểm định
print("File:", XLSX)

## Bước 2 — Khởi tạo Spark & đọc Excel (giữ đúng thứ tự dòng)

In [ ]:
import unicodedata, re, os, pandas as pd
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType, LongType
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

spark = (SparkSession.builder.appName("GraceSweetDemand").master("local[*]")
         .config("spark.sql.shuffle.partitions","8").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

pdf = pd.read_excel(XLSX)[["STT","Ngày tạo đơn","Sản phẩm"]]
pdf.columns=["stt","ngay","sp"]; pdf["row_idx"]=range(len(pdf))
rows=[(int(i),(None if pd.isna(n) else str(n)),(None if pd.isna(s) else str(s)))
      for i,n,s in zip(pdf["row_idx"],pdf["ngay"],pdf["sp"])]
schema=StructType([StructField("row_idx",LongType()),StructField("ngay",StringType()),StructField("sp",StringType())])
sdf=spark.createDataFrame(rows,schema)
print("Đã đọc:", sdf.count(), "dòng")

## Bước 3 — Điền ngày xuống từng dòng sản phẩm
File xuất theo đơn: chỉ dòng đầu mỗi đơn có ngày, các dòng sản phẩm phía dưới để trống.
Ta 'điền xuôi' (forward-fill) ngày của đơn xuống mọi dòng sản phẩm thuộc đơn đó.

In [ ]:
wq=Window.orderBy("row_idx").rowsBetween(Window.unboundedPreceding,0)
sdf=(sdf.withColumn("ngay",F.when(F.trim("ngay")=="",None).otherwise(F.col("ngay")))
        .withColumn("ngay",F.last("ngay",ignorenulls=True).over(wq))
        .withColumn("dt",F.to_date("ngay","dd/MM/yyyy")).filter(F.col("sp").isNotNull()))
print("Có ngày hợp lệ:", sdf.filter(F.col('dt').isNotNull()).count())

## Bước 4 — Tách mã hàng: style / màu / size
Xử lý 2 khuôn chính: `TÊN / Màu: X / Size: Y` (Facebook) và `... / màu / size` (Shopee).
Giữ **COD** là một mã riêng (hàng chưa gán mã). Chuẩn hoá Unicode NFC + chữ HOA.

In [ ]:
def parse(s):
    if s is None: return ("?",None,None,"NA")
    s=unicodedata.normalize("NFC",str(s)).strip()
    if s=="": return ("?",None,None,"NA")
    if s.upper().replace(" ","")=="COD": return ("COD",None,None,"COD")
    def cs(x):
        if x is None: return None
        x=x.strip().upper().replace("SIZE","").strip().lstrip(":").strip().replace("FREESIZE","FREE")
        x=re.split(r"\s",x)[0] if x else x
        return x or None
    m=re.search(r"^(.*?)\s*/\s*màu\s*:\s*(.*?)(?:\s*/\s*size\s*:\s*(.*?))?\s*$",s,flags=re.IGNORECASE)
    if m: return (m.group(1).strip().upper(),(m.group(2) or "").strip().upper() or None,cs(m.group(3)),"A")
    if s.count("/")>=1:
        p=[x.strip() for x in s.split("/")]
        size=cs(p[-1]) if len(p)>=2 else None
        color=p[-2].strip().upper() if len(p)>=3 else None
        nm=re.sub(r"^\d{6,}\s*-\s*","",p[0]); mc=re.search(r"\(([^)]+)\)",nm)
        code=mc.group(1).strip().upper() if mc else None
        nm=re.sub(r"^[^-]*-\s*\([^)]*\)\s*-?\s*","",nm).strip(); nm=re.sub(r"\(([^)]+)\)","",nm).strip()
        return (code or (nm.upper() if nm else s.upper()),color,size,"B")
    return (s.upper(),None,None,"OTHER")

psch=StructType([StructField("style",StringType()),StructField("color",StringType()),
                 StructField("size",StringType()),StructField("pat",StringType())])
u=F.udf(parse,psch)
sdf=(sdf.withColumn("p",u("sp")).select("dt",F.col("p.style").alias("style"),
        F.col("p.color").alias("color"),F.col("p.size").alias("size"))
        .filter(F.col("dt").isNotNull())
        .withColumn("week",F.to_date(F.date_trunc("week","dt")))).cache()
print("Dòng:",sdf.count()," | Tuần:",sdf.select('week').distinct().count(),
      " | Style:",sdf.select('style').distinct().count())

## Bước 5 — Doanh số theo tuần + chọn TOP style + điền tuần trống = 0
Giả định **1 dòng = 1 sản phẩm bán ra** (file không có cột số lượng).

In [ ]:
weekly=sdf.groupBy("style","week").agg(F.count(F.lit(1)).alias("y"))
top=[r[0] for r in sdf.groupBy("style").count().orderBy(F.desc("count")).limit(TOP_N).collect()]
weeks=[r["week"] for r in sdf.select("week").distinct().orderBy("week").collect()]
spine=spark.createDataFrame([(s,w) for s in top for w in weeks],["style","week"])
panel=spine.join(weekly,["style","week"],"left").fillna({"y":0})
print("TOP style:",len(top)," | số tuần:",len(weeks))

## Bước 6 — Tạo đặc trưng chuỗi thời gian
Lag (1,2,4 tuần), trung bình trượt (4,8 tuần), tuần trong năm, tháng,
cờ Tết, cờ tháng khuyến mãi (9–12), và xu hướng.

In [ ]:
w=Window.partitionBy("style").orderBy("week")
panel=(panel.withColumn("lag1",F.lag("y",1).over(w)).withColumn("lag2",F.lag("y",2).over(w))
       .withColumn("lag4",F.lag("y",4).over(w))
       .withColumn("roll4",F.avg("y").over(w.rowsBetween(-4,-1)))
       .withColumn("roll8",F.avg("y").over(w.rowsBetween(-8,-1)))
       .withColumn("woy",F.weekofyear("week")).withColumn("month",F.month("week"))
       .withColumn("is_tet",(F.month("week")==2).cast("int"))
       .withColumn("is_promo",F.month("week").isin(9,10,11,12).cast("int"))
       .withColumn("trend",F.row_number().over(w))
       .dropna(subset=["lag1","lag2","lag4","roll4","roll8"]))

## Bước 7 — Chia dữ liệu theo thời gian (12 tuần cuối = kiểm định)

In [ ]:
cut=weeks[-TEST_WEEKS]; train=panel.filter(F.col("week")<cut); test=panel.filter(F.col("week")>=cut)
print("Train:",train.count()," | Test:",test.count()," | Cutoff:",cut)

## Bước 8 — Huấn luyện GBTRegressor + so sánh 2 baseline
Baseline: **Naive** (bằng tuần trước) và **Trung bình trượt 4 tuần**. Đo RMSE, MAE, sMAPE.

In [ ]:
feats=["lag1","lag2","lag4","roll4","roll8","woy","month","is_tet","is_promo","trend","styleIdx"]
model=Pipeline(stages=[
    StringIndexer(inputCol="style",outputCol="styleIdx",handleInvalid="keep"),
    VectorAssembler(inputCols=feats,outputCol="features"),
    GBTRegressor(labelCol="y",featuresCol="features",maxIter=60,maxDepth=4,stepSize=0.1,seed=42)
]).fit(train)
pred=model.transform(test).withColumn("gbt",F.greatest("prediction",F.lit(0.0)))

def metrics(df,pcol):
    e=df.select(F.col("y").cast("double"),F.col(pcol).cast("double").alias("p"))
    rmse=RegressionEvaluator(labelCol="y",predictionCol="p",metricName="rmse").evaluate(e)
    mae =RegressionEvaluator(labelCol="y",predictionCol="p",metricName="mae").evaluate(e)
    smape=e.select((F.avg(F.when((F.abs("y")+F.abs("p"))==0,0.0)
            .otherwise(2*F.abs(F.col("p")-F.col("y"))/(F.abs("y")+F.abs("p"))))*100).alias("s")).first()["s"]
    return rmse,mae,smape
res={"GBT":metrics(pred,"gbt"),"Naive (tuần trước)":metrics(test,"lag1"),"Trung bình trượt 4T":metrics(test,"roll4")}
print("===== SAI SỐ TRÊN 12 TUẦN KIỂM ĐỊNH =====")
for k,(r,m,s) in res.items(): print(f"{k:22s} RMSE={r:6.2f}  MAE={m:6.2f}  sMAPE={s:5.1f}%")
best=min(res,key=lambda k:res[k][1]); print(">> Mô hình MAE thấp nhất:",best)

## Bước 9 — Dự báo tuần tới bằng mô hình tốt nhất & phân bổ màu/size
Dự báo cấp *style* → phân bổ xuống *màu/size* theo tỷ lệ 8 tuần gần nhất (dự báo phân cấp).

In [ ]:
last=weeks[-1]; base=panel.filter(F.col("week")==last)
if best=="GBT":
    fut=(base.withColumn("lag1",F.col("y")).withColumn("trend",F.col("trend")+1)
         .withColumn("week",F.date_add("week",7)).withColumn("woy",F.weekofyear("week"))
         .withColumn("month",F.month("week")).withColumn("is_tet",(F.month("week")==2).cast("int"))
         .withColumn("is_promo",F.month("week").isin(9,10,11,12).cast("int")))
    fc=model.transform(fut).select("style",F.round(F.greatest("prediction",F.lit(0))).alias("fc_style"))
elif best=="Naive (tuần trước)":
    fc=base.select("style",F.round("y").alias("fc_style"))
else:
    fc=base.select("style",F.round("roll4").alias("fc_style"))

recent=sdf.filter(F.col("week")>=weeks[-8]).filter(F.col("style").isin(top))
share=(recent.groupBy("style","color","size").count()
       .withColumn("tot",F.sum("count").over(Window.partitionBy("style")))
       .withColumn("share",F.col("count")/F.col("tot")))
plan=(share.join(fc,"style").withColumn("cut_qty",F.round(F.col("share")*F.col("fc_style")))
      .filter(F.col("cut_qty")>0).select("style","color","size","cut_qty").orderBy(F.desc("cut_qty")))
plan.show(15,truncate=38)

## Bước 10 — Lưu & tải kết quả về máy

In [ ]:
pd.DataFrame([(k,)+v for k,v in res.items()],columns=["Mô hình","RMSE","MAE","sMAPE"]).to_excel(f"{OUT}/ket_qua_sai_so.xlsx",index=False)
plan.toPandas().to_excel(f"{OUT}/ke_hoach_cat_tuan_toi.xlsx",index=False)
try:
    from google.colab import files
    files.download(f"{OUT}/ket_qua_sai_so.xlsx"); files.download(f"{OUT}/ke_hoach_cat_tuan_toi.xlsx")
except Exception as e: print("Bỏ qua tải tự động:",e)
print("Xong. Dòng kế hoạch cắt:",plan.count())
spark.stop()

## Ghi chú diễn giải (đưa vào Chương 4)
- Trên dữ liệu ~14 tháng, đơn tín hiệu, rời rạc: **naive/trung bình trượt mạnh hơn GBT** về MAE/RMSE. Đây là kết quả hợp lệ, đúng lý thuyết (ML dự báo phát huy khi có dữ liệu đa biến, khối lượng lớn).
- Giá trị của pipeline: (1) **kế hoạch cắt theo màu/size** mà baseline không tự sinh được; (2) khả năng mở rộng và sẵn sàng khai thác tín hiệu giàu hơn (số lượng, mã sản phẩm thay cho COD, dữ liệu nhiều năm, khuyến mãi/thời tiết).
- Giới hạn ghi rõ trong bài: (1) 1 dòng = 1 sản phẩm; (2) ~10,9% doanh số là mã COD chưa gán; (3) chỉ ~1 chu kỳ mùa vụ.